In [10]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

DATA = "../data/processed/openai_discharge_note_parses/azure_gpt56_luna_full_160095_20260713/openai_discharge_visit_parse_results.csv"

In [11]:
df = pd.read_csv(DATA)

df.head()

,sequence_index,subject_id,hadm_id,note_count,note_ids,note_types,note_seqs,note_charttimes,index_time,index_time_source,...,treatment__tevar_or_endovascular_repair,treatment__medical_management_only,treatment__prior_repair_or_stent_graft_only_no_new_repair,treatment__repaired_this_admission,treatment__evidence,control_usefulness__primary_discharge_diagnosis_or_alternative_explanation,control_usefulness__plausible_aortic_dissection_mimic_presentation,control_usefulness__evidence,overall__target_cleanup_label,overall__brief_summary
0,20,18110461,20001947,1,18110461-DS-8,DS,8,2138-11-08 00:00:00,2138-11-06 18:39:00,edregtime,...,no,no,no,no,Major Surgical or Invasive Procedure: none. Tr...,Multifocal bacterial pneumonia causing fever a...,yes,Discharge Diagnosis: multifocal pneumonia; ple...,no_aortic_syndrome_evidence,The admission was for fever and pleuritic ches...
1,95,17256683,20006820,1,17256683-DS-22,DS,22,2189-07-04 00:00:00,2189-07-01 20:55:00,edregtime,...,no,no,no,no,Major Surgical or Invasive Procedure: None.,Recurrent C. difficile infection causing diarr...,no,Discharge Diagnosis: PRIMARY Recurrent CDiff; ...,no_aortic_syndrome_evidence,Admission was for recurrent C. difficile diarr...
2,8,17913090,20000351,1,17913090-DS-13,DS,13,2145-06-18 00:00:00,2145-06-13 00:43:00,edregtime,...,no,no,no,no,Major Surgical or Invasive Procedure: None.,Acute psychosis/schizoaffective disorder and u...,no,Discharge Diagnosis: Primary: Acute psychosis;...,no_aortic_syndrome_evidence,This admission was for acute psychosis and unc...
3,90,18203391,20006478,1,18203391-DS-29,DS,29,2139-04-29 00:00:00,2139-02-21 10:38:00,edregtime,...,no,no,no,no,"Major Surgical or Invasive Procedure: ""None""",Intractable epilepsy with increased seizure fr...,no,"""admitted ... for increased frequency of seizu...",no_aortic_syndrome_evidence,This admission was for increased seizure frequ...
4,37,10289011,20003008,1,10289011-DS-12,DS,12,2134-07-04 00:00:00,2134-06-29 22:26:00,admittime,...,no,no,no,no,"Major Surgical or Invasive Procedure: ""None""; ...",Postoperative acute kidney injury on chronic k...,no,"""ATN seems like a reasonable etiology""; ""No ac...",no_aortic_syndrome_evidence,Admission was for postoperative acute on chron...


In [13]:
sum_categories = df['overall__target_cleanup_label'].value_counts()

print(sum_categories)

overall__target_cleanup_label
no_aortic_syndrome_evidence         133810
aneurysm_without_dissection          13279
dissection_ruled_out                  8929
aortic_syndrome_not_dissection        2819
prior_or_chronic_dissection_only       895
confirmed_acute_dissection             351
unclear                                 10
Name: count, dtype: int64


In [21]:
unique_dissection_patients = df[df['overall__target_cleanup_label'] == 'confirmed_acute_dissection']['subject_id'].nunique()

print(f"Number of unique dissection patients from parsing: {unique_dissection_patients}")

df_prev = pd.read_csv("../data/processed/pipeline_cache/raw_matrix.csv")
ICD_dissection_patients = df_prev[df_prev["is_aortic_dissection"].eq(1) & df_prev["ecg_count"].fillna(0).ge(1)]
print(f"Number of previously identified target patients: {ICD_dissection_patients['subject_id'].nunique()}")

overlap = set(ICD_dissection_patients['subject_id']).intersection(set(df[df['overall__target_cleanup_label'] == 'confirmed_acute_dissection']['subject_id']))
print(f"Number of overlapping patients: {len(overlap)}")

Number of unique dissection patients from parsing: 325
Number of previously identified target patients: 355
Number of overlapping patients: 251


In [20]:
print(type(unique_dissection_patients))

<class 'int'>


In [15]:
num_potential_controls = df[df['control_usefulness__plausible_aortic_dissection_mimic_presentation'] == 'yes'].shape[0]

print(num_potential_controls)

19344


## Target-discrepancy diagnostics

The cells below compare the full discharge-note parse against the prior ICD-defined target cohort from `raw_matrix.csv`. They keep subject-level and admission-level comparisons separate, because a large part of the discrepancy can come from comparing patient counts to encounter counts.


In [ ]:
from pathlib import Path

PARSE_RESULTS = Path(DATA)
RUN_DIR = PARSE_RESULTS.parent
PARSE_ERRORS = RUN_DIR / "openai_discharge_visit_parse_errors.jsonl"
RAW_MATRIX = Path("../data/processed/pipeline_cache/raw_matrix.csv")
DIAGNOSES_ICD = Path("../data/mimic-iv/hosp/diagnoses_icd.csv")
D_ICD_DIAGNOSES = Path("../data/mimic-iv/hosp/d_icd_diagnoses.csv")

TARGET_LABEL = "confirmed_acute_dissection"
TARGET_COL = "overall__target_cleanup_label"

parse_cols = [
    "subject_id",
    "hadm_id",
    "note_count",
    "note_ids",
    "index_time",
    "index_time_source",
    "first_ecg_time_24h",
    "ecg_count_24h",
    TARGET_COL,
    "overall__brief_summary",
    "aortic_syndrome_status__confirmed_acute_aortic_dissection",
    "aortic_syndrome_status__chronic_or_prior_dissection_only",
    "aortic_syndrome_status__aneurysm_without_dissection",
    "aortic_syndrome_status__dissection_explicitly_ruled_out",
    "diagnosis_context__diagnosed_before_arrival_at_outside_hospital",
    "diagnosis_context__diagnostic_modality",
    "anatomic_detail__stanford_type",
    "treatment__repaired_this_admission",
    "aortic_syndrome_status__evidence",
]
parse_cols = [c for c in parse_cols if c in df.columns]

parsed = df.copy()
parsed["subject_id"] = pd.to_numeric(parsed["subject_id"], errors="coerce").astype("Int64")
parsed["hadm_id"] = pd.to_numeric(parsed["hadm_id"], errors="coerce").astype("Int64")
parsed[TARGET_COL] = parsed[TARGET_COL].astype("string")
parsed["parsed_confirmed_acute_dissection"] = parsed[TARGET_COL].eq(TARGET_LABEL)

prev = pd.read_csv(
    RAW_MATRIX,
    usecols=["subject_id", "index_hadm_id", "cohort_split", "is_aortic_dissection", "ecg_count"],
)
prev["subject_id"] = pd.to_numeric(prev["subject_id"], errors="coerce").astype("Int64")
prev["index_hadm_id"] = pd.to_numeric(prev["index_hadm_id"], errors="coerce").astype("Int64")
prev["is_aortic_dissection"] = pd.to_numeric(prev["is_aortic_dissection"], errors="coerce").fillna(0).astype(int)

prev_targets_all = prev[prev["is_aortic_dissection"].eq(1)].copy()
prev_targets_ecg_complete = prev_targets_all[prev_targets_all["ecg_count"].fillna(0).ge(1)].copy()
parsed_confirmed = parsed[parsed["parsed_confirmed_acute_dissection"]].copy()

print(f"Parsed result rows: {len(parsed):,}")
print(f"Parsed unique subjects: {parsed['subject_id'].nunique():,}")
print(f"Parsed confirmed acute-dissection rows: {len(parsed_confirmed):,}")
print(f"Parsed confirmed acute-dissection subjects: {parsed_confirmed['subject_id'].nunique():,}")
print(f"Prior broad ICD target subjects/admissions: {prev_targets_all['subject_id'].nunique():,} / {prev_targets_all['index_hadm_id'].nunique():,}")
print(f"Prior ECG-complete ICD target subjects/admissions: {prev_targets_ecg_complete['subject_id'].nunique():,} / {prev_targets_ecg_complete['index_hadm_id'].nunique():,}")
if PARSE_ERRORS.exists():
    print(f"Parse error records: {sum(1 for line in PARSE_ERRORS.read_text().splitlines() if line.strip()):,}")


In [ ]:
def overlap_summary(name, prior_df):
    prior_subjects = set(prior_df["subject_id"].dropna().astype(int))
    prior_hadms = set(prior_df["index_hadm_id"].dropna().astype(int))
    parsed_subjects = set(parsed_confirmed["subject_id"].dropna().astype(int))
    parsed_hadms = set(parsed_confirmed["hadm_id"].dropna().astype(int))
    return {
        "prior_definition": name,
        "prior_subjects": len(prior_subjects),
        "prior_hadms": len(prior_hadms),
        "parsed_confirmed_subjects": len(parsed_subjects),
        "parsed_confirmed_hadms": len(parsed_hadms),
        "subject_overlap": len(prior_subjects & parsed_subjects),
        "prior_subjects_not_parsed_confirmed": len(prior_subjects - parsed_subjects),
        "parsed_confirmed_subjects_not_prior": len(parsed_subjects - prior_subjects),
        "hadm_overlap": len(prior_hadms & parsed_hadms),
        "prior_hadms_not_parsed_confirmed": len(prior_hadms - parsed_hadms),
        "parsed_confirmed_hadms_not_prior": len(parsed_hadms - prior_hadms),
    }

pd.DataFrame([
    overlap_summary("broad ICD target cohort in raw_matrix", prev_targets_all),
    overlap_summary("active ECG-complete target subset", prev_targets_ecg_complete),
])


In [ ]:
# What did the parser call the previous ECG-complete ICD targets?
prior_ecg_with_parse = prev_targets_ecg_complete.merge(
    parsed[parse_cols],
    left_on=["subject_id", "index_hadm_id"],
    right_on=["subject_id", "hadm_id"],
    how="left",
    indicator=True,
)

print("Prior ECG-complete target admissions found in parsed run:")
print(prior_ecg_with_parse["_merge"].value_counts(dropna=False))
print("\nParser target_cleanup_label distribution among prior ECG-complete targets:")
display(prior_ecg_with_parse[TARGET_COL].fillna("<missing_from_parse>").value_counts(dropna=False).to_frame("n"))

review_cols = [
    "subject_id", "index_hadm_id", "cohort_split", "hadm_id", TARGET_COL,
    "aortic_syndrome_status__confirmed_acute_aortic_dissection",
    "aortic_syndrome_status__chronic_or_prior_dissection_only",
    "aortic_syndrome_status__aneurysm_without_dissection",
    "aortic_syndrome_status__dissection_explicitly_ruled_out",
    "diagnosis_context__diagnostic_modality",
    "anatomic_detail__stanford_type",
    "treatment__repaired_this_admission",
    "overall__brief_summary",
    "aortic_syndrome_status__evidence",
]
review_cols = [c for c in review_cols if c in prior_ecg_with_parse.columns]

prior_ecg_not_confirmed = prior_ecg_with_parse[
    ~prior_ecg_with_parse[TARGET_COL].eq(TARGET_LABEL)
].copy()
print(f"Prior ECG-complete targets not confirmed by parsed note: {len(prior_ecg_not_confirmed):,}")
display(prior_ecg_not_confirmed[review_cols].head(50))


In [ ]:
# Compare parsed-confirmed admissions with exact aortic-dissection ICD codes.
# This uses the current exact-code rule: normalize by removing decimals; no broad prefix matching.
TARGET_ICD9 = {"441", "44100", "44101", "44103"}
TARGET_ICD10 = {"I7100", "I7101", "I71010", "I71012", "I71019", "I7103"}

def normalize_icd_code(s):
    return s.astype("string").str.upper().str.replace(".", "", regex=False).str.strip()

hadms_to_audit = set(parsed["hadm_id"].dropna().astype(int)) | set(prev_targets_all["index_hadm_id"].dropna().astype(int))
parts = []
for chunk in pd.read_csv(DIAGNOSES_ICD, usecols=["subject_id", "hadm_id", "icd_code", "icd_version"], chunksize=500_000):
    chunk["hadm_id"] = pd.to_numeric(chunk["hadm_id"], errors="coerce").astype("Int64")
    chunk = chunk[chunk["hadm_id"].isin(hadms_to_audit)].copy()
    if chunk.empty:
        continue
    chunk["norm_icd_code"] = normalize_icd_code(chunk["icd_code"])
    chunk["icd_version"] = pd.to_numeric(chunk["icd_version"], errors="coerce").astype("Int64")
    chunk["is_exact_ad_icd"] = (
        (chunk["icd_version"].eq(9) & chunk["norm_icd_code"].isin(TARGET_ICD9))
        | (chunk["icd_version"].eq(10) & chunk["norm_icd_code"].isin(TARGET_ICD10))
    )
    parts.append(chunk)

dx = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
exact_ad_dx = dx[dx["is_exact_ad_icd"]].copy()
exact_codes_by_hadm = (
    exact_ad_dx.groupby("hadm_id")["norm_icd_code"]
    .agg(lambda codes: "|".join(sorted(set(codes.dropna()))))
    .rename("exact_ad_icd_codes")
    .reset_index()
)

parsed_audit = parsed.merge(exact_codes_by_hadm, on="hadm_id", how="left")
parsed_audit["has_exact_ad_icd"] = parsed_audit["exact_ad_icd_codes"].notna()

print(f"Audited diagnosis rows for parsed/prior hadm_ids: {len(dx):,}")
print(f"Admissions with exact AD ICD code in audited set: {exact_ad_dx['hadm_id'].nunique():,}")
print("\nParsed-confirmed by exact ICD-code status:")
display(pd.crosstab(parsed_audit["parsed_confirmed_acute_dissection"], parsed_audit["has_exact_ad_icd"], margins=True))

parsed_confirmed_without_exact_icd = parsed_audit[
    parsed_audit["parsed_confirmed_acute_dissection"] & ~parsed_audit["has_exact_ad_icd"]
].copy()
exact_icd_not_parsed_confirmed = parsed_audit[
    parsed_audit["has_exact_ad_icd"] & ~parsed_audit["parsed_confirmed_acute_dissection"]
].copy()

print(f"Parsed-confirmed admissions without exact target ICD code: {len(parsed_confirmed_without_exact_icd):,}")
print(f"Exact target-ICD admissions in parsed run not parsed-confirmed: {len(exact_icd_not_parsed_confirmed):,}")


In [ ]:
# Review parsed-confirmed admissions that were not in the prior ECG-complete model target set.
prior_ecg_hadms = set(prev_targets_ecg_complete["index_hadm_id"].dropna().astype(int))
parsed_confirmed_not_prior_ecg = parsed_audit[
    parsed_audit["parsed_confirmed_acute_dissection"]
    & ~parsed_audit["hadm_id"].astype("Int64").isin(prior_ecg_hadms)
].copy()

print(f"Parsed-confirmed admissions not in prior ECG-complete target set: {len(parsed_confirmed_not_prior_ecg):,}")
print("Breakdown by exact target ICD code presence:")
display(parsed_confirmed_not_prior_ecg["has_exact_ad_icd"].value_counts(dropna=False).to_frame("n"))

cols = [
    "subject_id", "hadm_id", "has_exact_ad_icd", "exact_ad_icd_codes", TARGET_COL,
    "diagnosis_context__diagnostic_modality", "anatomic_detail__stanford_type",
    "treatment__repaired_this_admission", "overall__brief_summary",
    "aortic_syndrome_status__evidence",
]
cols = [c for c in cols if c in parsed_confirmed_not_prior_ecg.columns]
display(parsed_confirmed_not_prior_ecg[cols].head(50))


In [ ]:
# Top non-target ICD diagnoses among note-confirmed dissection admissions without an exact AD ICD code.
# Useful for spotting miscoding, postoperative/history-only cases, or alternate aortic-syndrome coding patterns.
if not parsed_confirmed_without_exact_icd.empty and not dx.empty:
    titles = pd.read_csv(D_ICD_DIAGNOSES)
    titles["norm_icd_code"] = normalize_icd_code(titles["icd_code"])
    titles["icd_version"] = pd.to_numeric(titles["icd_version"], errors="coerce").astype("Int64")
    no_icd_hadms = set(parsed_confirmed_without_exact_icd["hadm_id"].dropna().astype(int))
    non_target_dx = dx[dx["hadm_id"].isin(no_icd_hadms) & ~dx["is_exact_ad_icd"]].copy()
    top_non_target_codes = (
        non_target_dx.groupby(["icd_version", "norm_icd_code"], as_index=False)
        .agg(admissions=("hadm_id", "nunique"))
        .sort_values("admissions", ascending=False)
        .head(50)
        .merge(
            titles[["icd_version", "norm_icd_code", "long_title"]].drop_duplicates(),
            on=["icd_version", "norm_icd_code"],
            how="left",
        )
    )
    display(top_non_target_codes)
else:
    print("No parsed-confirmed admissions without exact target ICD codes, or no diagnosis rows loaded.")


In [ ]:
# Subject-level view: patients can have multiple parsed admissions.
subject_parse_summary = (
    parsed.groupby("subject_id")
    .agg(
        parsed_admissions=("hadm_id", "nunique"),
        parsed_confirmed_admissions=("parsed_confirmed_acute_dissection", "sum"),
        labels=(TARGET_COL, lambda x: "|".join(sorted(set(x.dropna().astype(str))))),
    )
    .reset_index()
)
subject_parse_summary["any_parsed_confirmed"] = subject_parse_summary["parsed_confirmed_admissions"].gt(0)

prior_ecg_subjects = set(prev_targets_ecg_complete["subject_id"].dropna().astype(int))
subject_parse_summary["in_prior_ecg_target_subjects"] = subject_parse_summary["subject_id"].astype(int).isin(prior_ecg_subjects)

subject_discrepancy_summary = pd.crosstab(
    subject_parse_summary["in_prior_ecg_target_subjects"],
    subject_parse_summary["any_parsed_confirmed"],
    rownames=["in_prior_ecg_target_subjects"],
    colnames=["any_parsed_confirmed"],
    margins=True,
)
display(subject_discrepancy_summary)

prior_subjects_without_any_parsed_confirmation = subject_parse_summary[
    subject_parse_summary["in_prior_ecg_target_subjects"] & ~subject_parse_summary["any_parsed_confirmed"]
].copy()
parsed_confirmed_subjects_not_prior = subject_parse_summary[
    ~subject_parse_summary["in_prior_ecg_target_subjects"] & subject_parse_summary["any_parsed_confirmed"]
].copy()

print(f"Prior ECG target subjects without any parsed-confirmed admission: {len(prior_subjects_without_any_parsed_confirmation):,}")
print(f"Parsed-confirmed subjects not in prior ECG target subjects: {len(parsed_confirmed_subjects_not_prior):,}")
display(prior_subjects_without_any_parsed_confirmation.head(50))
display(parsed_confirmed_subjects_not_prior.head(50))


In [ ]:
# Optional: write compact review tables for manual chart/label adjudication.
review_dir = RUN_DIR / "target_discrepancy_review"
review_dir.mkdir(exist_ok=True)

prior_ecg_not_confirmed[review_cols].to_csv(review_dir / "prior_ecg_targets_not_parsed_confirmed.csv", index=False)
parsed_confirmed_not_prior_ecg[cols].to_csv(review_dir / "parsed_confirmed_not_prior_ecg_targets.csv", index=False)
parsed_confirmed_without_exact_icd[cols].to_csv(review_dir / "parsed_confirmed_without_exact_target_icd.csv", index=False)
exact_icd_not_parsed_confirmed[parse_cols + ["exact_ad_icd_codes"]].to_csv(review_dir / "exact_target_icd_not_parsed_confirmed.csv", index=False)

print(f"Wrote review tables to: {review_dir}")
